# PhysREVE — Baseline Comparison

Ablation table isolating three factors:

| Model | Pretraining | Physics | Expected |
|---|---|---|---|
| LDA | — | — | lowest |
| Logistic Regression | — | — | low |
| Random-init PhysREVE | none | arch only | mid |
| Base REVE | MAE only | none | higher |
| **PhysREVE** | MAE + L_phys + L_snr | full | highest |

Swap the synthetic data block for your real dataset loader.

In [4]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    # Colab already ships torch, numpy, scipy, sklearn, matplotlib, seaborn, tqdm, requests
    # Install only what's missing
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

In [5]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from physreve import (
    PhysREVEConfig,
    LabeledEEGDataset, make_split_loaders,
    run_baseline_finetune, run_mae_pretraining, run_pretraining, run_finetuning,
    extract_features, run_ml_baselines,
    compare_models,
)
from physreve.physics import build_leadfield, motor_roi_indices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

ImportError: cannot import name 'run_mae_pretraining' from 'physreve' (/content/PhysREVE/physreve/__init__.py)

## 1. Data
Replace this cell with your real dataset (BCI IV 2a, CHB-MIT, etc.).
Expected shape: `X` → `(N, C, T)`, `y` → `(N,)` integer labels.

In [ ]:
# ── Synthetic stand-in (22-ch, 4-class, 1s @ 250 Hz) ──────────────────────
np.random.seed(42)
N_TRIALS = 200
N_CH     = 22
SFREQ    = 250
T        = SFREQ          # 1 second
N_CLASS  = 4

X = np.random.randn(N_TRIALS, N_CH, T).astype(np.float32)
y = np.repeat(np.arange(N_CLASS), N_TRIALS // N_CLASS)

# Train / val / test split via make_split_loaders
train_loader, val_loader, test_loader = make_split_loaders(
    X, y, train_frac=0.70, val_frac=0.15, batch_size=16, seed=42
)

# Numpy arrays for ML baselines
tr_idx  = train_loader.dataset.indices if hasattr(train_loader.dataset, 'indices') else range(len(train_loader.dataset))
val_idx = val_loader.dataset.indices   if hasattr(val_loader.dataset,  'indices') else range(len(val_loader.dataset))

X_np = (X - X.mean(-1, keepdims=True)) / X.std(-1, keepdims=True).clip(1e-8)
X_tr_np,  y_tr_np  = X_np[list(tr_idx)],  y[list(tr_idx)]
X_val_np, y_val_np = X_np[list(val_idx)], y[list(val_idx)]

print(f'Train: {X_tr_np.shape}  Val: {X_val_np.shape}')

## 2. Leadfield & Config
Replace `ch_names` with your actual channel list.

In [ ]:
# Standard 10-20 names for 22-ch BCI IV 2a
ch_names = [
    'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
    'P1', 'Pz', 'P2', 'POz'
]

L_col_np, L_row_np, src_pos, info = build_leadfield(ch_names, sfreq=SFREQ)

L_col = torch.tensor(L_col_np, dtype=torch.float32).to(device)
L_row = torch.tensor(L_row_np, dtype=torch.float32).to(device)

# Electrode positions (metres)
elec_xyz = torch.tensor(
    np.array([info['chs'][i]['loc'][:3] for i in range(N_CH)]),
    dtype=torch.float32
).to(device)

# Motor ROI indices for asymmetry loss
lh_idx_np, rh_idx_np = motor_roi_indices(info, src_pos, ch_names)
lh_idx = torch.tensor(lh_idx_np).to(device)
rh_idx = torch.tensor(rh_idx_np).to(device)

print(f'Leadfield: {L_col_np.shape}  |  LH sources: {len(lh_idx)}  RH: {len(rh_idx)}')

In [ ]:
cfg = PhysREVEConfig(
    d_model=256, n_heads=8, n_layers=6,   # n_layers=6 for quick experiments
    patch_size=50,
    n_sources=L_col.shape[1],
    lambda_phys=0.15, lambda_snr=0.05, lambda_asym=0.05,
    dropout=0.1,
)
print(cfg)

## 3. ML Baselines (LDA, Logistic Regression)

In [ ]:
ml_results = run_ml_baselines(X_tr_np, y_tr_np, X_val_np, y_val_np, sfreq=SFREQ)
print('\nML baseline val accuracies:', ml_results)

## 4. Random-Init PhysREVE (no pretraining)

In [ ]:
rand_model, rand_hist = run_baseline_finetune(
    cfg, train_loader, val_loader,
    L_row=L_row, L_col=L_col, elec_xyz=elec_xyz,
    device=device, n_classes=N_CLASS,
    n_epochs=30, lr=3e-4,
    lh_idx=lh_idx, rh_idx=rh_idx,
)
rand_val_acc = max(rand_hist['val_acc'])
print(f'Random-init best val acc: {rand_val_acc:.3f}')

## 5. Base REVE (MAE pretraining, no physics)

In [ ]:
from torch.utils.data import DataLoader
from physreve.data import UnlabeledEEGDataset

# Use training split as unlabeled corpus (replace with larger unlabeled set if available)
unlabeled_loader = DataLoader(
    UnlabeledEEGDataset(X_tr_np), batch_size=16, shuffle=True
)

reve_pretrained, reve_pre_hist = run_mae_pretraining(
    cfg, unlabeled_loader, L_row=L_row, elec_xyz=elec_xyz,
    device=device, n_epochs=20, lr=3e-4,
)

In [ ]:
reve_model, reve_ft_hist = run_finetuning(
    reve_pretrained, cfg, train_loader, val_loader,
    L_col=L_col, elec_xyz=elec_xyz, device=device,
    n_classes=N_CLASS, n_epochs=30,
    lh_idx=lh_idx, rh_idx=rh_idx,
)
reve_val_acc = max(reve_ft_hist['val_acc'])
print(f'Base REVE best val acc: {reve_val_acc:.3f}')

## 6. PhysREVE (full)

In [ ]:
phys_pretrained, phys_pre_hist = run_pretraining(
    cfg, unlabeled_loader,
    L_row=L_row, L_col=L_col, elec_xyz=elec_xyz,
    device=device, n_epochs=20, lr=3e-4, sfreq=SFREQ,
)

In [ ]:
phys_model, phys_ft_hist = run_finetuning(
    phys_pretrained, cfg, train_loader, val_loader,
    L_col=L_col, elec_xyz=elec_xyz, device=device,
    n_classes=N_CLASS, n_epochs=30,
    lh_idx=lh_idx, rh_idx=rh_idx,
)
phys_val_acc = max(phys_ft_hist['val_acc'])
print(f'PhysREVE best val acc: {phys_val_acc:.3f}')

## 7. Results

In [ ]:
results = {
    **{k: v for k, v in ml_results.items()},
    'random_init': rand_val_acc,
    'base_reve':   reve_val_acc,
    'physreve':    phys_val_acc,
}

print('\n=== Ablation Table ===')
print(f'{"Model":<20}  {"Val Acc":>8}')
print('-' * 32)
for name, acc in results.items():
    print(f'{name:<20}  {acc:>8.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart — final val accs
ax = axes[0]
names = list(results.keys())
accs  = list(results.values())
colors = ['#aaa', '#aaa', '#6baed6', '#2171b5', '#08306b']
bars = ax.bar(names, accs, color=colors[:len(names)])
ax.set_ylim(0, 1)
ax.set_ylabel('Val Accuracy')
ax.set_title('Ablation: val accuracy')
ax.tick_params(axis='x', rotation=20)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

# Learning curves — fine-tuning val acc
ax = axes[1]
ax.plot(rand_hist['val_acc'],     label='Random init', linestyle='--', color='#6baed6')
ax.plot(reve_ft_hist['val_acc'],  label='Base REVE',   linestyle='-.',  color='#2171b5')
ax.plot(phys_ft_hist['val_acc'],  label='PhysREVE',    linestyle='-',   color='#08306b')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy')
ax.set_title('Fine-tuning learning curves')
ax.legend()

plt.tight_layout()
plt.savefig('ablation_results.png', dpi=120)
plt.show()